In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim

from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)


def start(mission: MissionBase):
    sim = CarSim(prop, mission)
    success = sim.run()
    if success:
        SimDrawer(sim).show()

# ミッション4

ロボットをコースに沿って移動させ、コース外周に設けられた標識（original）をみつけたらコースから外れて標識の真上で1秒以上停車し、その後、コースに復帰してからスタート地点まで戻ろう。
- コースに沿って走らせるにはauto命令を使えるが、auto命令は車がコースの中にいて、かつコースの向きと車の向きがあっているときのみ使えることに注意しよう（コースに対して斜めの状態でONにしても上手く走れない）
- コースに復帰する為に複数の種類の標識を自由に道路においてもよい。うまく誘導しよう。

## 取り組み方
1. 使える命令を理解する
2. 下のセルを実行して、ロボットの限界速度や、ロボットが存在する初期位置を把握する
3. ２つ下のセル内において
    - プログラムを書く
    - 路面標識を好きな位置に設置する
    - 実行して結果を見る
4. シミュレータでうまく動いたらロボットにプログラムを書きこんで動かしてみよう（講師に声をかけてね）

|使える命令|意味|指定できる値|使い方の例|
|--|--|--|--|
|move|一定速度で前に進む|v=速度[m/秒]|move(v=0.2)|
|move|一定時間だけ一定速度で前に進む|v=速度[m/秒], t=時間[秒]|move(v=0.2, t=1.0)|
|rotate|一定速度で回転する。w>0は左回転。w<0は右回転|w=回転速度[度/秒]|rotate(w=90)|
|rotate|一定時間だけ一定速度で回転する。w>0は左回転。w<0は右回転|w=回転速度[度/秒], t=時間[秒]|rotate(w=90, t=1.0)|
|wait|直前の命令が終わるまで待つ||wait()|
|serach|標識を見つける（複数見つかった場合は、最も近いもの）||pos = search()|
|serach|特定の標識を見つける（複数見つかった場合は、最も近いもの）|name = "標識名"|pos = search(name="stop")|
|auto|コースに沿って進む|v=速度[m/秒]|auto(v=0.2)|

## 注意点
- スタート時の位置はランダムに最大10cmほどずれる
- スタート時の向きはランダムに最大5度ほどずれる

In [ ]:
class Mission4Base(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=80)
        self.goals = [
            GoalCircle((2.2, 1.6), 0.2, should_stop=True),
            GoalCircle((2.5, 0.0), 0.2, should_stop=False),
        ]
        self.initial_xy = (2.5, 0.0)
        self.random_d_xy = (0.1, 0.1)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=2.2, y=1.6, name="original"),
            ]
        )

print("最大速度", prop.max_velocity, "m/秒")
print("最大回転速度", prop.max_rotate_deg, "度/秒")
MissionDrawer(Mission4Base()).show()


In [ ]:
class Mission4(Mission4Base):
    def __init__(self):
        super().__init__()
        self.set_signs(
            [
                Sign(x=2, y=2, name="original"),
                ### 標識を追加するにはここから下を書き換える
                Sign(x=1.4, y=0.9, name="stop"),
                Sign(x=0.8, y=0.8, name="warn"),
                ### 標識を追加するにはここより上を書き換える
            ]
        )

    @staticmethod
    def command_func(alive, *, move, rotate, wait, search, auto, **kwargs):
        ######## ここから下にプログラムを書こう
        mode = 0
        while alive():
            if mode == 0:
                # コース内で自動走行し、標識(original)を見つけた場合に標識の真上に行き１秒停車するモード
                pos = search(name="original")
                if pos is None:
                    auto(v=0.2)
                else:
                    if pos.theta > 5:
                        rotate(w=45)
                    elif pos.theta < -5:
                        rotate(w=-45)
                    else:
                        move(v=0.2)
                        move(v=0.2, t=pos.x / 0.2)
                        wait()
                        move(v=0, t=1)
                        wait()
                        mode = 1
            elif mode == 1:
                # 左回転し、標識(stop)を見つけた場合に標識の真上に行くモード
                pos = search(name="stop")
                if pos is None:
                    rotate(w=45)
                else:
                    if pos.theta > 5:
                        rotate(w=45)
                    elif pos.theta < -5:
                        rotate(w=-45)
                    else:
                        move(v=0.2)
                        move(v=0.2, t=pos.x / 0.2)
                        wait()
                        mode = 2
            elif mode == 2:
                # 右回転し、標識(warn)を見つけた場合に標識の方を向いた後で自動走行を開始するモード
                pos = search(name="warn")
                if pos is None:
                    rotate(w=-45)
                else:
                    rotate(w=pos.theta, t=1)
                    wait()
                    auto(v=0.2)
                    mode = 0

        ######## ここより上にプログラムを書こう


start(Mission4())

# ミッション4のヒント
- 標識（original）をみつけたらコースから外れて標識に向かうまで
    - コースの外にでるには、autoでなくmoveを使う必要がある
    - 標識（original）を見つけたら、「moveやrotateを使用して標識の真上まで進む」というプログラムを書こう
    - 「標識の真上まで進む」には・・・
        - searchの結果得られる角度（pos.theta）を使ってrotateを行い標識の方を向こう
        - searchの結果得られる前方位置（pos.x）を使ってmoveとwaitを行い標識まで進もう
    - １秒停止するには、move(v=0,t=1)とwait()を使おう
- コースに復帰する
    - コースに復帰して自動走行を再開するには、車をレーン内でかつ、レーンに平行にする必要がある。この誘導の為に、標識（original）以外に標識を２種類使用する。例えば・・・
        - 標識(stop): レーンへの復帰位置（レーン内）に置く 
        - 標識(warn): レーンへの復帰位置からみて進むべき方向に置く
    - このように標識を置けば、下記で復帰できる
        - 標識(stop)まで進んだ後、標識(warn)の方向を向き、auto命令を実行する
    - 下記のように動作モードを定義して、それぞれプログラムすると見通しがよい
        - mode0 : コース内で自動走行し、標識(original)を見つけた場合に標識の真上に行き１秒停車するモード
        - mode1 : 左回転し、標識(stop)を見つけた場合に標識の真上に行くモード
        - mode2 : 右回転し、標識(warn)を見つけた場合に標識の方を向いた後で自動走行を開始するモード
        ```
        mode = 0
        while alive():
            if mode == 0:
                pos = search(name="original")
                if pos is None:
                    auto(v=0.2)
                else:
                    # ここに標識の真上に行き１秒停車するプログラムを書く
                    mode = 1
            elif mode == 1:
                pos = search(name="stop")
                if pos is None:
                    rotate(w=45)
                else:
                    # ここに標識の真上に行くプログラムを書く
                    mode = 2
            elif mode == 2:
                pos = search(name="warn")
                if pos is None:
                    rotate(w=-45)
                else:
                    # 標識の方を向いた後で自動走行を開始するプログラムを書く
                    mode = 0
        ```